In [1]:
import mysql.connector
import pandas as pd
from getpass import getpass

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter MySQL password: "),
    database="tmdb_movie_analytics",
    port=3306
)

cursor = connection.cursor()

print("SQL Analysis notebook connected successfully!")

SQL Analysis notebook connected successfully!


In [3]:
query_1 = """
SELECT DISTINCT
    m.movie_id,
    m.title,
    m.release_date,
    g.genre_name
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
AND YEAR(m.release_date) > 2015
AND m.status = 'Released'
ORDER BY m.release_date DESC;
"""

cursor.execute(query_1)

results = cursor.fetchall()

columns = [column[0] for column in cursor.description]

action_movies = pd.DataFrame(results, columns=columns)

print("Total Action movies released after 2015:", len(action_movies))

display(action_movies.head(10))

Total Action movies released after 2015: 268


,movie_id,title,release_date,genre_name
0,931285,Mortal Kombat II,2026-05-06,Action
1,1242898,Predator: Badlands,2025-11-05,Action
2,617126,The Fantastic 4: First Steps,2025-07-23,Action
3,1061474,Superman,2025-07-09,Action
4,911430,F1,2025-06-25,Action
5,1234821,Jurassic World Rebirth,2025-06-23,Action
6,1087192,How to Train Your Dragon,2025-06-06,Action
7,541671,Ballerina,2025-06-04,Action
8,575265,Mission: Impossible - The Final Reckoning,2025-05-17,Action
9,986056,Thunderbolts*,2025-04-30,Action


In [4]:
query_2 = """
SELECT
    m.title,
    m.revenue,
    GROUP_CONCAT(
        DISTINCT g.genre_name
        ORDER BY g.genre_name
        SEPARATOR ', '
    ) AS genres
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE m.revenue > 0
AND m.status = 'Released'
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
"""

cursor.execute(query_2)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
top_grossing_movies = pd.DataFrame(results, columns=columns)

display(top_grossing_movies)

,title,revenue,genres
0,Avatar,2.923706e+09,"Action, Adventure, Science Fiction"
1,Avengers: Endgame,2.799439e+09,"Action, Adventure, Science Fiction"
2,Avatar: The Way of Water,2.334485e+09,"Action, Adventure, Science Fiction"
3,Titanic,2.264162e+09,"Drama, Romance"
4,Star Wars: The Force Awakens,2.068224e+09,"Action, Adventure, Science Fiction"
5,Avengers: Infinity War,2.052415e+09,"Action, Adventure, Science Fiction"
6,Spider-Man: No Way Home,1.921207e+09,"Action, Adventure, Science Fiction"
7,Zootopia 2,1.868209e+09,"Adventure, Animation, Comedy, Family, Mystery"
8,Inside Out 2,1.698864e+09,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1.671537e+09,"Adventure, Science Fiction, Thriller"


In [5]:
query_3 = """
SELECT
    g.genre_name,
    ROUND(AVG(m.budget), 2) AS average_budget,
    ROUND(AVG(m.revenue), 2) AS average_revenue,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE m.status = 'Released'
AND m.budget > 0
AND m.revenue > 0
GROUP BY g.genre_name
ORDER BY average_revenue DESC;
"""

cursor.execute(query_3)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
genre_financials = pd.DataFrame(results, columns=columns)

display(genre_financials)

,genre_name,average_budget,average_revenue,movie_count
0,Adventure,1.113933e+08,4.014512e+08,655
1,Animation,8.828229e+07,3.821035e+08,231
2,Family,8.840877e+07,3.578708e+08,329
3,Science Fiction,9.786440e+07,3.356897e+08,446
4,Action,9.465142e+07,3.091258e+08,784
5,Fantasy,8.918367e+07,3.079130e+08,378
6,Comedy,5.465252e+07,2.230400e+08,721
7,Music,4.281702e+07,2.058438e+08,47
8,Romance,3.722695e+07,1.772269e+08,320
9,Thriller,5.084428e+07,1.700493e+08,635


In [6]:
query_4 = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast
WHERE actor_name IS NOT NULL
AND actor_name != ''
GROUP BY person_id, actor_name
ORDER BY movie_count DESC
LIMIT 10;
"""

cursor.execute(query_4)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
top_actors = pd.DataFrame(results, columns=columns)

display(top_actors)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Willem Dafoe,32
7,Scarlett Johansson,32
8,Tom Cruise,31
9,Morgan Freeman,31


In [7]:
query_5 = """
SELECT
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
AND person_name IS NOT NULL
AND person_name != ''
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC, director_name;
"""

cursor.execute(query_5)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
top_directors = pd.DataFrame(results, columns=columns)

display(top_directors)

,director_name,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
...,...,...
212,Stephen Sommers,4
213,Tate Taylor,4
214,Tim Johnson,4
215,Tim Story,4


In [8]:
query_6 = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
WHERE keyword_name IS NOT NULL
AND keyword_name != ''
GROUP BY keyword_id, keyword_name
ORDER BY movie_count DESC
LIMIT 10;
"""

cursor.execute(query_6)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
top_keywords = pd.DataFrame(results, columns=columns)

display(top_keywords)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [9]:
query_7 = """
SELECT
    title,
    budget,
    revenue
FROM movies
WHERE status = 'Released'
AND budget > 0
AND revenue > 0
AND budget > (
    SELECT AVG(budget)
    FROM movies
    WHERE status = 'Released'
    AND budget > 0
)
AND revenue < (
    SELECT AVG(revenue)
    FROM movies
    WHERE status = 'Released'
    AND revenue > 0
)
ORDER BY budget DESC;
"""

cursor.execute(query_7)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
high_budget_low_revenue = pd.DataFrame(results, columns=columns)

print("Number of movies:", len(high_budget_low_revenue))
display(high_budget_low_revenue.head(10))

Number of movies: 273


,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,Green Lantern,200000000.0,219851172.0
4,Mulan,200000000.0,69965374.0
5,Jungle Cruise,200000000.0,220889446.0
6,Wonder Woman 1984,200000000.0,169601036.0
7,Killers of the Flower Moon,200000000.0,158722599.0
8,Onward,200000000.0,141888276.0
9,Luca,200000000.0,51074773.0


In [10]:
query_8 = """
SELECT DISTINCT
    c.actor_name
FROM cast AS c
JOIN crew AS cr
    ON c.movie_id = cr.movie_id
WHERE cr.person_name = 'Christopher Nolan'
AND cr.job = 'Director'
AND c.actor_name IS NOT NULL
AND c.actor_name != ''
ORDER BY c.actor_name;
"""

cursor.execute(query_8)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
nolan_actors = pd.DataFrame(results, columns=columns)

print("Number of actors:", len(nolan_actors))
display(nolan_actors)

Number of actors: 84


,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


In [11]:
query_9 = """
SELECT
    g.genre_name,
    ROUND(AVG(m.vote_average), 2) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE m.status = 'Released'
AND m.vote_count > 0
GROUP BY g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""

cursor.execute(query_9)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
highest_rated_genre = pd.DataFrame(results, columns=columns)

display(highest_rated_genre)

,genre_name,average_rating,movie_count
0,War,7.36,85


In [12]:
query_10 = """
SELECT
    m.title,
    COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies AS m
JOIN movie_keywords AS mk
    ON m.movie_id = mk.movie_id
WHERE m.status = 'Released'
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC, m.title
LIMIT 10;
"""

cursor.execute(query_10)
results = cursor.fetchall()

columns = [column[0] for column in cursor.description]
keyword_rich_movies = pd.DataFrame(results, columns=columns)

display(keyword_rich_movies)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,13 Hours: The Secret Soldiers of Benghazi,49
9,How to Train Your Dragon: The Hidden World,49
